---



# Fra Folketingets referater til analysérbare data

**Social Data Science 1: lektion 7**

Denne notebook viser fuld preprocessering af dataen anvendt i lektion 7, fra den rå JSON-fil til det datasæt, vi analyserer.

Hvert trin er en beslutning. Nogle af dem er tekniske, men de fleste handler om, hvad vi mener med *en politisk debat*. Læg mærke til, hvad der ændre sig undervejs

**Rådata:** `dkparl_resumes_2007-2022.json`. Referater af møderne i Folketinget, oktober 2007 til november 2022, hentet fra ft.dk.

## Anvend notebooken

Hvert punkt følger samme form:

1. **Læs** hvad trinnet skal gøre, og hvorfor.
2. **Kør** koden. Der sker som regel kun én ting per celle.
3. **Åbn data** i [viewer].
4. **Kig efter** hvad har ændret sig siden sidst?

---

## 1. Indlæs rådata

Først henter vi de to biblioteker, vi skal bruge. `json` kan læse JSON-filer, og `pandas`
laver dem om til tabeller.

In [1]:
import json
import pandas as pd

Stien til filen. Ret den, så den peger på din egen kopi.

In [2]:
sti = "/Users/jeppefl/Library/CloudStorage/OneDrive-AalborgUniversitet/01_work/01_undervisning/02_sds1/03_data/dkparl/dkparl_resumes_2007-2022.json"

`dtype=False` gør, at pandas ikke selv prøver at gætte typerne.

In [3]:
raw = pd.read_json(sti, dtype=False)
d = raw.copy()

`pd.DataFrame()` laver listen om til en tabel, så hvert element bliver en række, og hver nøgle i dictionary'en bliver en kolonne.

In [4]:
d = pd.DataFrame(d)

print("Antal elementer:", len(d))
print(d.columns.tolist())

Antal elementer: 1711
['Title', 'SubTitle', 'Filename', 'MetaMeeting-tingdokID', 'ParliamentarySession', 'ParliamentarySession-tingdokID', 'ParliamentaryGroup', 'ParliamentaryGroup-tingdokID', 'MeetingNumber', 'DateOfSitting', 'Location', 'EdixiDocLocation', 'AudioFileFolder', 'AgendaLong', 'AgendaShort', 'Items', 'ns0:ParliamentarySession-tingdokID', 'ns0:ParliamentaryGroup-tingdokID']


**Kig efter i viewer:** 
Én række per møde. Find kolonnen `Items`. Hvad står der i cellerne? Det er ikke et tal eller
en tekst, men en hel dictionary, der er pakket ind i én celle.

---

## 2. Ryd op i kolonnerne

Mange kolonner er tekniske id'er og filstier fra Folketingets system. Vi skal ikke bruge dem, så vi fjerner dem, før vi går videre. Det gør tabellen lettere at overskue. Det er dog ikke en (analytisk) nødvendighed. 

In [7]:
df = d.drop(columns=["MetaMeeting-tingdokID",
                       "ParliamentarySession-tingdokID",
                       "ParliamentaryGroup",
                       "EdixiDocLocation",
                       "AudioFileFolder",
                       "ns0:ParliamentarySession-tingdokID",
                       "ns0:ParliamentaryGroup-tingdokID",
                       "MeetingNumber",
                       "Title",
                       "Filename"])

print(df.columns.tolist())

['SubTitle', 'ParliamentarySession', 'ParliamentaryGroup-tingdokID', 'DateOfSitting', 'Location', 'AgendaLong', 'AgendaShort', 'Items']


**Kig efter i viewer:** 
Stadig én række per møde, men færre kolonner. `DateOfSitting` er mødets dato, og
`ParliamentarySession` er folketingsåret.

---

## 3. Fold dagsordenspunkterne ud

### 3.1 Hvad ligger der i `Items`?

Vi tager ét møde ud og kigger ind i cellen:

In [8]:
foerste_moede = df.loc[1, "Items"]

print(type(foerste_moede))
print(foerste_moede.keys())

<class 'dict'>
dict_keys(['0', '1', '2'])


Nøglerne er `"1"`, `"2"`, `"3"` ... altså dagsordenspunkternes numre. Hvert punkt er igen en
dictionary med sine egne felter:

In [9]:
print(foerste_moede["1"].keys())

dict_keys(['ItemNo', 'ItemTitle', 'ItemStartDateTime', 'EndDateTime', 'ItemText'])


Vi vil have **én række per dagsordenspunkt**, ikke per møde. Det er samme idé som med ordene i
lektion 6: gør feltet til en liste, og brug `.explode()`.

### 3.2 Tæl først

Før vi ændrer noget, tæller vi, hvor mange punkter der er. Så kan vi bagefter tjekke, at
tallene går op. `.map(len)` giver antallet af punkter i hvert møde.

In [10]:
antal_i_hvert_moede = df["Items"].map(len)

print("Punkter i alt:", antal_i_hvert_moede.sum())
print("Møder uden punkter:", (antal_i_hvert_moede == 0).sum())

Punkter i alt: 27106
Møder uden punkter: 2


### 3.3 Gør punkterne til en liste

Vi tager kun de kolonner med, vi skal bruge. `.copy()` giver en selvstændig tabel, så vi ikke
kommer til at ændre i `df`.

In [13]:
punkter = df[["DateOfSitting", "ParliamentarySession", "Items"]].copy()

`dict.values` henter værdierne ud af hver dictionary (og smider nøglerne `"1"`, `"2"` ... væk).
`list` gør dem til en almindelig liste, som `.explode()` kan arbejde med.

In [14]:
punkter["Items"] = punkter["Items"].map(dict.values).map(list)

**Kig efter i viewer:** 
Stadig én række per møde. Men `Items` er nu en liste af punkter i stedet for en dictionary.

### 3.4 Fold ud

`.explode()` giver hvert element i listen sin egen række. De andre kolonner bliver gentaget.
`ignore_index=True` nummererer rækkerne 0, 1, 2 ... forfra.

In [15]:
print("Rækker før explode:", len(punkter))

punkter = punkter.explode("Items", ignore_index=True)

print("Rækker efter explode:", len(punkter))

Rækker før explode: 1711
Rækker efter explode: 27108


**Kig efter i viewer:** 
Nu er der én række per dagsordenspunkt. Samme dato står i mange rækker efter hinanden, fordi
et møde har mange punkter. Hver celle i `Items` er nu én dictionary.

### 3.5 Tjek: går tallene op?

Sammenlign antallet af rækker med tallet fra 3.2. Der er to rækker for meget.

De to møder uden punkter havde en tom liste, og `.explode()` gør hver tom liste til en række
med `NaN`. Det giver ingen fejlbesked. Man opdager det kun ved at tjekke, at tallene går op.

In [16]:
print("Rækker uden punkt:", punkter["Items"].isna().sum())

Rækker uden punkt: 2


Vi fjerner dem. `reset_index(drop=True)` nummererer rækkerne 0, 1, 2 ... igen, så der ikke er
huller. Det får betydning i næste trin.

In [17]:
punkter = punkter.dropna(subset=["Items"]).reset_index(drop=True)

print("Dagsordenspunkter:", len(punkter))

Dagsordenspunkter: 27106


### 3.6 Pak felterne ud til kolonner

Hvert punkt er stadig en dictionary inde i én celle. `pd.json_normalize()` laver en liste af
dictionaries om til en tabel, hvor hver nøgle bliver en kolonne.

In [18]:
felter = pd.json_normalize(punkter["Items"].tolist())

print(felter.columns.tolist())

['ItemNo', 'ItemTitle', 'ItemStartDateTime', 'EndDateTime', 'ItemText']


**Kig efter i viewer:** 
Samme antal rækker som `punkter`, men nu med én kolonne per felt: titel, tekst, start- og
sluttidspunkt osv. Datoen og folketingsåret mangler, for de lå ikke inde i punkterne.

Til sidst sætter vi de to tabeller sammen side om side. `.drop(columns="Items")` fjerner den
gamle kolonne, og `.join()` sætter `felter` på, række for række.

`.join()` matcher rækkerne på deres nummer. Det virker, fordi både `punkter` (efter
`reset_index` i 3.5) og `felter` er nummereret 0, 1, 2 ...

In [19]:
punkter = punkter.drop(columns="Items").join(felter)

print("Dagsordenspunkter:", len(punkter))
print(punkter.columns.tolist())

Dagsordenspunkter: 27106
['DateOfSitting', 'ParliamentarySession', 'ItemNo', 'ItemTitle', 'ItemStartDateTime', 'EndDateTime', 'ItemText']


**Kig efter i viewer:** 
Én række per dagsordenspunkt med dato, folketingsår og alle punktets felter som almindelige
kolonner. Det er den form, vi kan arbejde videre med.

---

## 4. Gør tidspunkterne til rigtige tider

`ItemStartDateTime` og `EndDateTime` ligner tidspunkter, men pandas læser dem som tekst:

In [20]:
print(punkter["ItemStartDateTime"].dtype)
print(punkter["ItemStartDateTime"].iloc[0])

object
2008-11-04T13:00:47


`object` (eller `str` i nyere pandas) betyder tekst. Formatet `2008-11-04T13:00:47` er dato, et `T` og klokkeslæt.
`pd.to_datetime()` kender formatet og laver det om til rigtige tidspunkter, som man kan regne
med.

In [21]:
punkter["ItemStartDateTime"] = pd.to_datetime(punkter["ItemStartDateTime"])
punkter["EndDateTime"] = pd.to_datetime(punkter["EndDateTime"])

print(punkter["ItemStartDateTime"].dtype)

datetime64[ns]


Nu kan vi bruge `.dt` til at trække dele ud, på samme måde som `.str` bruges til tekst.
`strftime("%H:%M")` skriver klokkeslættet som tekst med timer og minutter.

In [22]:
punkter["start"] = punkter["ItemStartDateTime"].dt.strftime("%H:%M")
punkter["slut"] = punkter["EndDateTime"].dt.strftime("%H:%M")

To tidspunkter kan trækkes fra hinanden. Resultatet er en varighed, som vi omregner til
minutter.

In [23]:
varighed = punkter["EndDateTime"] - punkter["ItemStartDateTime"]
punkter["varighed_min"] = (varighed.dt.total_seconds() / 60).round(1)

**Kig efter i viewer:** 
Rul til højre og find `start`, `slut` og `varighed_min`. Sortér efter `varighed_min`. Hvad er
de korteste og de længste punkter? Er der nogen, der ser forkerte ud?

`start` og `slut` er tekst til at læse. Skal man regne på tiderne, bruger man de rigtige
tidskolonner.

---

## 5. Vælg 1. behandling af lovforslag

### 5.1 Hvad slags punkter er der?

Ikke alle dagsordenspunkter er debatter. Titlen fortæller, hvilken slags punkt det er:

In [24]:
print(punkter["ItemTitle"].head(20))

0                                               Punkt 0
1              2. behandling af L 31: Om sundhedsloven.
2     2. behandling af L 37: Om Den Særlige Pensions...
3     1. behandling af B 15: Om Albaniens og Kroatie...
4     1. behandling af B 24: Om den internationale s...
5                                               Punkt 0
6                                               Punkt 0
7     2. behandling af L 169: Om agentur for Retshån...
8                                               Punkt 0
9                                               Punkt 0
10    Spørgsmål til ministrene til umiddelbar besvar...
11                                      Spm. nr. US 109
12                                      Spm. nr. US 110
13                                      Spm. nr. US 111
14                                      Spm. nr. US 112
15                                      Spm. nr. US 113
16                                      Spm. nr. US 114
17                                      Spm. nr.

Vi trækker punktets type ud af begyndelsen af titlen med et regex. Læst i bidder:

| Del | Betyder |
|---|---|
| `^` | begyndelsen af titlen |
| `\d\. behandling af [LB]` | *1.*, *2.* eller *3. behandling af* efterfulgt af L (lovforslag) eller B (beslutningsforslag) |
| lodret streg | eller |
| `Forhandling af [FVR]` | *Forhandling af* efterfulgt af F, V eller R |
| lodret streg | eller |
| `Valg` | ordet *Valg* |

In [25]:
typer = r"^(\d\. behandling af [LB]|Forhandling af [FVR]|Valg)"

punkter["type"] = punkter["ItemTitle"].str.extract(typer, expand=False)

Titler, der ikke passer på mønsteret, får `NaN`. Hvor mange er der?

In [26]:
print("Uden type:", punkter["type"].isna().sum())

Uden type: 14650


Dem kalder vi "Andet":

In [27]:
punkter["type"] = punkter["type"].fillna("Andet")

print(punkter["type"].value_counts())

type
Andet                 14650
3. behandling af L     3227
1. behandling af L     3143
2. behandling af L     2989
1. behandling af B     2264
Forhandling af F        544
Forhandling af R        168
Valg                    116
2. behandling af B        5
Name: count, dtype: int64


### 5.2 Hvor lange er punkterne?

Vi måler hvert punkts tekst i antal tegn. `fillna("")` gør manglende tekster til tom tekst, så
de tæller som 0 tegn.

In [28]:
punkter["laengde"] = punkter["ItemText"].fillna("").str.len()

Og så antal punkter og mediantal tegn for hver type:

In [29]:
oversigt = (punkter
            .groupby("type")["laengde"]
            .agg(antal="size", median_tegn="median")
            .sort_values("antal", ascending=False))

print(oversigt)

                    antal  median_tegn
type                                  
Andet               14650       4886.0
3. behandling af L   3227        809.0
1. behandling af L   3143      29046.0
2. behandling af L   2989        802.0
1. behandling af B   2264      50640.5
Forhandling af F      544      95191.0
Forhandling af R      168     107056.0
Valg                  116        859.0
2. behandling af B      5        751.0


"2. og 3. behandlinger" er typisk 800 tegn, fordi det er en afstemning og et par formalia. Den politiske debat om et lovforslag foregår ved **1. behandlingen**. Det er dem, vi beholder.

*Beslutning:* Vi fravælger beslutningsforslag (B), forespørgsler (F) og alt andet. Det er lovgivningen, vi studerer, ikke Folketinget som helhed.

In [30]:
forste = punkter[punkter["type"] == "1. behandling af L"].copy()

print("1. behandling af lovforslag:", len(forste))

1. behandling af lovforslag: 3143


### 5.3 Sæt kolonnerne i en meningsfuld rækkefølge

Vi sætter de kolonner, vi kigger mest på, forrest: hvornår, hvad og hvor langt. Resten
kommer bagefter i samme rækkefølge som før.

In [31]:
forrest = ["DateOfSitting", "ParliamentarySession", "ItemTitle", "type",
           "start", "slut", "varighed_min", "laengde", "ItemText"]

resten = forste.columns.drop(forrest).tolist()

forste = forste[forrest + resten]

**Kig efter i viewer:** 
Kun 1. behandlinger. Læg mærke til rækkenumrene til venstre, som springer, fordi vi har
filtreret rækker fra. Klik på en celle i `ItemText` og læs begyndelsen af en debat. Hvem har
fremsat lovforslaget?

---

## 6. Find ministeren

Datasættet har ingen emnevariabel. Men hvert lovforslag er fremsat af en minister, og det
står i indledningen af hvert punkt:

In [32]:
print(forste["ItemText"].iloc[10][:800])

16) 1. behandling af lovforslag nr. L 190:
Forslag til lov om ændring af straffeloven, lov om forbud mod ophold i bestemte ejendomme og lov om fuldbyrdelse af straf m.v. (Styrket indsats mod rocker- og bandekriminalitet m.v.).
Af justitsministeren (Søren Pape Poulsen).
(Fremsættelse 26.04.2017).
Forhandlingen er åbnet. Det er fru Trine Bramsen, Socialdemokratiet. Værsgo.
Det er helt urimeligt, når kriminelle rocker- og bandegrupperinger tror, at de skal styre og bestemme i vores samfund; når de omgår loven igen og igen, når de truer hjemløse, afpresser forretningsdrivende, skyder i gaderne og skaber utryghed.
Desværre så vi sidste år en stor stigning i antallet af skyderier i rocker- og bandemiljøerne i Odense, i Københavnsområdet, på Sydsjælland og flere andre steder. Vi så konflikterne m


Vi trækker ministeren ud med et regulært udtryk. Læst i bidder:

| Del | Betyder |
|---|---|
| `\nAf ` | et linjeskift efterfulgt af *Af* |
| `(.*?minister\w*` | tekst frem til et ord med *minister* ... |
| `(?: for [^(\n]+?)?)` | ... eventuelt efterfulgt af *for noget* |
| `\s*\(` | og stop ved parentesen med ministerens navn |

Vi kigger kun i de første 900 tegn, altså indledningen. Ellers kunne en minister, der nævnes
senere i debatten, blive fanget ved en fejl.

In [34]:
regex = r"\nAf (.*?minister\w*(?: for [^(\n]+?)?)\s*\("

indledning = forste["ItemText"].str[:900]

forste["minister"] = indledning.str.extract(regex, expand=False)
forste["minister"] = forste["minister"].str.strip()

print("Minister fundet:", forste["minister"].notna().sum(), "af", len(forste))

Minister fundet: 3082 af 3143


**Kig efter i viewer:** 
Sortér efter `minister`. Bemærk forskellige stavemåder af det samme ministerium.

In [35]:
uden = forste[forste["minister"].isna()]

print(len(uden), "uden minister")
print()
print(uden["ItemText"].iloc[0][:300])

61 uden minister

4) 1. behandling af lovforslag nr. L 115:
Forslag til lov om ændring af lov om autorisation af sundhedspersoner og om sundhedsfaglig virksomhed. (Mulighed for, at ikkeautoriserede personer kan udføre priktest).
Af Liselott Blixt (DF) m.fl.
(Fremsættelse 05.02.2016).
Forhandlingen er åbnet. Den først


**Kig efter i viewer:** 
Læs indledningen i et par af rækkerne. Hvem har fremsat dem?

De er fremsat af **folketingsmedlemmer**, ikke af regeringen. Ikke alle lovforslag er
regeringens.

*Beslutning:* De får værdien "Andet" i stedet for `NaN`.

In [36]:
forste["minister"] = forste["minister"].fillna("Andet")

print("Uden minister nu:", forste["minister"].isna().sum())

Uden minister nu: 0


---

## 7. Saml ministrene i politikområder

Ministerierne skifter navn med hver regering. Samme ministerium kan hedde tre-fire ting
over femten år:

In [37]:
print(forste["minister"].value_counts().head(25))

minister
skatteministeren                            376
justitsministeren                           362
beskæftigelsesministeren                    306
undervisningsministeren                     122
miljøministeren                             110
transportministeren                         109
finansministeren                            104
erhvervsministeren                          101
erhvervs- og vækstministeren                 95
udlændinge- og integrationsministeren        90
kulturministeren                             75
økonomi- og erhvervsministeren               71
ministeren for sundhed og forebyggelse       70
Andet                                        61
børne- og undervisningsministeren            60
sundhedsministeren                           58
økonomi- og indenrigsministeren              57
fødevareministeren                           55
uddannelses- og forskningsministeren         50
miljø- og fødevareministeren                 49
social- og indenrigsministeren 

Vi samler titlerne i **seks politikområder** med en dictionary. Alt andet bliver
"Andet".

*Beslutninger:*

- Fødevareministeriet hører til **Miljø og klima**, fordi det i perioder var slået sammen
  med Miljøministeriet.
- *Ministeren for børn, ligestilling, integration og sociale forhold* bliver i **Andet**.       

In [38]:
omraader = {
    "Skat": ["skatteministeren"],
    "Retsvæsen": ["justitsministeren"],
    "Beskæftigelse": ["beskæftigelsesministeren"],
    "Uddannelse": ["undervisningsministeren",
                   "børne- og undervisningsministeren",
                   "uddannelses- og forskningsministeren",
                   "videnskabsministeren",
                   "ministeren for forskning, innovation og videregående uddannelser",
                   "ministeren for børn, undervisning og ligestilling"],
    "Miljø og klima": ["miljøministeren",
                       "miljø- og fødevareministeren",
                       "fødevareministeren",
                       "ministeren for fødevarer, landbrug og fiskeri",
                       "ministeren for fødevarer, fiskeri og ligestilling",
                       "klima- og energiministeren",
                       "klima-, energi- og bygningsministeren",
                       "klima-, energi- og forsyningsministeren",
                       "energi-, forsynings- og klimaministeren"],
    "Udlændinge": ["udlændinge- og integrationsministeren",
                   "integrationsministeren",
                   "udlændinge-, integrations- og boligministeren"],
}

### 7.1 Lav dictionary'en om til en opslagstabel

Dictionary'en har ét område per nøgle og en liste af titler som værdi. Vi vil have en tabel
med **én række per titel**. Her bruger vi igen `.explode()`:

In [39]:
opslag = pd.Series(omraader).explode()

print(opslag.head(10))

Skat                                               skatteministeren
Retsvæsen                                         justitsministeren
Beskæftigelse                              beskæftigelsesministeren
Uddannelse                                  undervisningsministeren
Uddannelse                        børne- og undervisningsministeren
Uddannelse                     uddannelses- og forskningsministeren
Uddannelse                                     videnskabsministeren
Uddannelse        ministeren for forskning, innovation og videre...
Uddannelse        ministeren for børn, undervisning og ligestilling
Miljø og klima                                      miljøministeren
dtype: object


Området står nu i indekset til venstre og titlen som værdi. `.reset_index()` gør indekset
til en almindelig kolonne, og så giver vi de to kolonner sigende navne.

In [40]:
opslag = opslag.reset_index()
opslag.columns = ["omraade", "minister"]

**Kig efter i viewer:** 
En lille tabel med to kolonner. Hver titel optræder én gang.

### 7.2 Slå området op

Nu kan vi koble opslagstabellen på `forste` via kolonnen `minister`. Det er en *left join*, hvor alle rækker i `forste` beholdes, og de får et område, hvis deres
minister står i opslagstabellen.

In [41]:
print("Rækker før:", len(forste))

forste = forste.merge(opslag, on="minister", how="left")

print("Rækker efter:", len(forste))

Rækker før: 3143
Rækker efter: 3143


Samme antal rækker før og efter. Det skal det være, da hver titel står kun én gang i opslagstabellen, så ingen debat bliver fordoblet.

Ministre, der ikke står i opslaget, har fået `NaN` som område. De bliver "Andet":

In [42]:
print("Uden område:", forste["omraade"].isna().sum())

forste["omraade"] = forste["omraade"].fillna("Andet")

print(forste["omraade"].value_counts())

Uden område: 1298
omraade
Andet             1298
Skat               376
Miljø og klima     372
Retsvæsen          362
Beskæftigelse      306
Uddannelse         275
Udlændinge         154
Name: count, dtype: int64


**Tjek:** hvilke titler er endt i "Andet"? Er der nogen, der burde høre til et af de
seks områder?

In [43]:
print(forste.loc[forste["omraade"] == "Andet", "minister"].value_counts().head(15))

minister
transportministeren                         109
finansministeren                            104
erhvervsministeren                          101
erhvervs- og vækstministeren                 95
kulturministeren                             75
økonomi- og erhvervsministeren               71
ministeren for sundhed og forebyggelse       70
Andet                                        61
sundhedsministeren                           58
økonomi- og indenrigsministeren              57
social- og indenrigsministeren               49
transport-, bygnings- og boligministeren     46
velfærdsministeren                           40
sundheds- og ældreministeren                 38
forsvarsministeren                           30
Name: count, dtype: int64


**Kig efter i viewer:** 
Kig på `omraade` og se, hvilke ministertitler der hører til hvert område.

---

## 8. Fjern indledningen

Indledningen af hver debat nævner lovforslagets titel **og ministeren**. Lader vi den blive stående, står svaret på vores klyngeanalyse i selve teksten, da debatter om skat indeholder
ordet *skatteministeren* i første linje.

Det hedder **label leakage**, hvilket betyder at etiketten/label lækker ind i det, man analyserer.

Debatten begynder efter linjen `(Fremsættelse dd.mm.åååå).` Vi deler teksten i to ved den linje. `n=1` betyder, at vi kun deler ved første forekomst.

In [44]:
dele = forste["ItemText"].str.split(r"\(Fremsættelse[^)]*\)\.?\n", n=1, regex=True)

Hver celle i `dele` er nu en liste. Er fremsættelseslinjen fundet, har listen to stykker: indledningen og debatten. Er den ikke fundet, har den kun ét stykke: hele teksten.

In [45]:
print(dele.str.len().value_counts())

ItemText
2    3117
1      26
Name: count, dtype: int64


`.str[-1]` tager det sidste stykke i hver liste. Det er debatten, hvis teksten blev delt, og
hele teksten, hvis den ikke blev.

In [46]:
forste["tekst"] = dele.str[-1]

print("FØR:")
print(forste["ItemText"].iloc[0][:500])
print()
print("EFTER:")
print(forste["tekst"].iloc[0][:500])

FØR:
9) 1. behandling af lovforslag nr. L 121:
Forslag til lov om ændring af lov om fremme af besparelser i energiforbruget, lov om varmeforsyning, lov om kommunal fjernkøling og forskellige andre love. (Implementering af EU’s energieffektivitetsdirektiv m.v.).
Af klima-, energi- og bygningsministeren (Martin Lidegaard).
(Fremsættelse 29.01.2014).
Forhandlingen er åbnet. Den første, der har ordet, er hr. Lars Christian Lilleholt, Venstre, som ordfører.
Først vil jeg godt ønske den nye minister tillyk

EFTER:
Forhandlingen er åbnet. Den første, der har ordet, er hr. Lars Christian Lilleholt, Venstre, som ordfører.
Først vil jeg godt ønske den nye minister tillykke med embedet. Jeg ser frem til et godt samarbejde.
Vi skal have lidt mere ro! Ellers er det ikke til for ordføreren at fortælle, hvad han gerne vil, og det er umuligt at høre, hvad der bliver sagt. Værsgo.
Først tillykke til den nye minister med embedet. Jeg ser frem til et godt samarbejde, om end jeg må konstatere, at på minis

**Kig efter i viewer:** 
Sammenlign begyndelsen af `ItemText` og `tekst` i et par rækker. Hvad er skåret væk?

---

## 9. Fjern de udgåede punkter

Nogle lovforslag blev taget af dagsordenen, men står stadig i referatet som et punkt. Vi
finder dem ved at se på de korte tekster.

In [47]:
tekstlaengde = forste["tekst"].str.len()

korte = forste[tekstlaengde < 2000]

print(len(korte), "debatter under 2.000 tegn")

73 debatter under 2.000 tegn


**Kig efter i viewer:** 
Læs `tekst` i nogle af rækkerne. Hvad står der?

In [48]:
print(korte["tekst"].str.contains("udgået af dagsordenen").sum(), "af dem er udgået")
print()
print(korte["tekst"].str.strip().value_counts().head(3))

31 af dem er udgået

tekst
(Punktet er udgået af dagsordenen).                            22
(Punktet er udgået af dagsordenen og overgået til møde 91).     3
(Punktet er udgået af dagsordenen og overgået til møde 69).     3
Name: count, dtype: int64


Lod vi dem blive, ville de være næsten tomme punkter i en klyngeanalyse. De ville ligge
langt fra alle andre og trække PCA'ens akser skævt.

*Beslutning:* Grænsen er 2.000 tegn. Det fjerner de udgåede punkter og en håndfuld meget
korte debatter, uden at ramme almindelige debatter, som typisk er 29.000 tegn.

In [49]:
forste = forste[tekstlaengde >= 2000].copy()

print("Debatter:", len(forste))
print(forste["omraade"].value_counts())

Debatter: 3070
omraade
Andet             1242
Skat               373
Miljø og klima     366
Retsvæsen          361
Beskæftigelse      301
Uddannelse         273
Udlændinge         154
Name: count, dtype: int64


---

## 10. Nye variable

År, lovforslagsnummer og et id, så hver debat kan genfindes. Én variabel ad gangen.

Datoen er de første 10 tegn af `DateOfSitting`:

In [50]:
forste["dato"] = forste["DateOfSitting"].str[:10]

Året er de første 4 tegn af datoen. `.astype(int)` gør det til et tal, så vi kan sortere og
regne med det.

In [51]:
forste["aar"] = forste["dato"].str[:4].astype(int)

Folketingsåret og titlen får kortere, danske navne:

In [52]:
forste["samling"] = forste["ParliamentarySession"]
forste["titel"] = forste["ItemTitle"]

Lovforslagets nummer, fx *L 12*, trækkes ud af titlen:

In [53]:
forste["lovnr"] = forste["ItemTitle"].str.extract(r"(L \d+)", expand=False)

Til sidst et id.

`reset_index(drop=True)` fjerner hullerne i rækkenumrene, og `.insert(0, ...)` sætter id'et
ind som den første kolonne.

In [54]:
forste = forste.reset_index(drop=True)
forste.insert(0, "id", range(1, len(forste) + 1))

print(forste[["id", "dato", "aar", "lovnr", "omraade"]].head())
print()
print(forste["aar"].value_counts().sort_index())

   id        dato   aar  lovnr         omraade
0   1  2014-02-06  2014  L 121  Miljø og klima
1   2  2018-11-15  2018   L 90           Andet
2   3  2015-05-08  2015  L 196   Beskæftigelse
3   4  2015-05-08  2015  L 197   Beskæftigelse
4   5  2015-05-08  2015  L 201           Andet

aar
2007     41
2008    223
2009    204
2010    217
2011    171
2012    229
2013    214
2014    185
2015    192
2016    205
2017    224
2018    235
2019    186
2020    226
2021    209
2022    109
Name: count, dtype: int64


**Kig efter i viewer:** 
`id` står forrest, og rækkenumrene går nu 0, 1, 2 ... uden huller. Sortér efter `aar`. Er der
lige mange debatter hvert år?

---

## 11. Gem datasættet

Vi gemmer kun de kolonner, analysen skal bruge. `ItemText` med indledningen er væk.

In [55]:
kolonner = ["id", "dato", "aar", "samling", "lovnr", "titel",
            "minister", "omraade", "tekst"]

lovforslag = forste[kolonner]

**Kig efter i viewer:** 
Det færdige datasæt: én række per debat, ni kolonner. Det er denne tabel, tekstanalysen
starter fra.

In [56]:
lovforslag.to_csv("ft_lovforslag.csv", index=False)

print(lovforslag.shape)

(3070, 9)


---

## Hvad har vi besluttet?

Intet af det følgende står i rådata. Det er valg, vi har truffet, og hvert af dem former det,
tekstanalysen kan finde.

| Trin | Beslutning | Hvad det betyder for analysen |
|---|---|---|
| 1 | Én række per dagsordenspunkt | Analyseenheden er et punkt, ikke et møde |
| 2 | Kun 1. behandling af lovforslag | Vi studerer lovgivning, ikke Folketinget som helhed |
| 3 | Ministeren findes med et regulært udtryk | En fejl i mønsteret flytter hele ministerier |
| 4 | Seks politikområder, resten er "Andet" | Områderne er vores definition, ikke Folketingets |
| 5 | Indledningen skæres væk | Ellers står "svaret" i teksten (label leakage) |
| 6 | Debatter under 2.000 tegn fjernes | Næsten tomme tekster ville forvride PCA |